# P1 · 03 Causal Cue Intervention Test and attribution

Algorithms 5 and 6: the four pre-specified image edits (T1 background transplant, T2 reverse transplant, T3 file re-encoding, T4 removal of extra-cranial pixels) with matched controls, paired t and sign-flip permutation tests with Holm correction, and Grad-CAM / Integrated Gradients attribution shares against brain and BRISC tumour masks (Tables 17–19, Supplementary Table S1, Figs. 9–10). Trains nothing: it reloads the checkpoints of notebook 01, so **run it after 01 has finished**. About 1.1 GPU hours on a T4. Needs the `KAGGLE_API_TOKEN` secret (the pipeline rebuilds its data and splits before the tests).

In [ ]:
# Setup: Google Drive, GPU check and code location
import os, sys, subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout or 'No GPU: Runtime > Change runtime type > T4 GPU')
# Location of this code package: either a GitHub clone or the folder uploaded to your Drive.
GITHUB_URL = 'https://github.com/IsaacJournals/brain-mri-source-shift-audit'   # set to '' to use a copy uploaded to MyDrive/P1_code_release instead
CODE_DIR = Path('/content/P1_code_release') if GITHUB_URL else Path('/content/drive/MyDrive/P1_code_release')
if GITHUB_URL and not CODE_DIR.exists():
    subprocess.run(['git', 'clone', '-q', GITHUB_URL, str(CODE_DIR)], check=True)
assert (CODE_DIR / 'src' / 'P1_pipeline.py').exists(), f'Code not found in {CODE_DIR}. Upload the P1_code_release folder to MyDrive or set GITHUB_URL.'
SRC = CODE_DIR / 'src'
print('code:', CODE_DIR)
# Kaggle credentials from Colab secrets, passed to the analysis process (values are never printed)
from google.colab import userdata
SECRET_ENV = {}
try:
    tok = userdata.get('KAGGLE_API_TOKEN')
    if tok: SECRET_ENV['KAGGLE_API_TOKEN'] = tok.strip()
except Exception:
    try:
        SECRET_ENV['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME'); SECRET_ENV['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except Exception:
        pass
print('Kaggle credentials found' if SECRET_ENV else 'No Kaggle secret: add KAGGLE_API_TOKEN (key icon, notebook access on)')

In [ ]:
# Run the causal-cue tests and attribution. Output is streamed here and written to MyDrive/MI10_P1/. Resume after a disconnect = Run all.
env = dict(os.environ, **SECRET_ENV, P1_MODE='FULL_PAPER_RUN', P1_IN_COLAB='1', P1_ADDON='1', P1_ADDON_PATH=str(SRC / 'P1_addon.py'))
p = subprocess.Popen([sys.executable, '-u', str(SRC / 'P1_pipeline.py')], env=env, cwd='/content',
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end='')
p.wait(); print('exit code', p.returncode)